# 06 — Performance Hospitalar: Quem Salva Mais Vidas?

A mortalidade por insuficiência respiratória (J96) varia enormemente entre hospitais. Parte dessa variação é explicada pelo perfil dos pacientes (idade, comorbidades, tipo de admissão). Este notebook separa **performance hospitalar** de **seleção de pacientes** usando ajuste de risco.

**Perguntas de Pesquisa:** RQ5 — Quais hospitais têm melhores desfechos ajustados por risco?

1. Qual a variação hospitalar em mortalidade bruta e ajustada?
2. Existe relação volume-desfecho (hospitais que tratam mais J96 têm menos mortes)?
3. Quais características do hospital (UTI, natureza jurídica, porte) predizem melhor performance?
4. Quais hospitais consistentemente sobre- ou sub-performam?

**Depende de:** `resp_failure_sih.parquet`, `hospital_tags.parquet`, `hospital_icu_beds.parquet`, `cnes_names.parquet`

In [1]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from shared import (
    load_resp_failure, load_hospital_tags, load_icu_beds, load_cnes_names, hospital_name,
    setup_plot_style, save_plot, save_metrics,
    COLORS, ERA_COLORS, ERA_LABELS, ERA_ORDER, COMORBIDITY_GROUPS, SECONDARY_DIAG_COLS,
    extract_comorbidities, classify_legal_nature,
)

setup_plot_style()
resp = load_resp_failure()
resp = resp[resp["year"] >= 2016].copy()
tags = load_hospital_tags()
icu_beds = load_icu_beds()
names_df = load_cnes_names()

resp = resp.merge(tags[["CNES", "icu_capacity_level", "broad_type", "nat_jur_category"]], on="CNES", how="left")
resp = resp.merge(icu_beds[["CNES", "icu_beds_sus", "total_beds"]], on="CNES", how="left")
resp["icu_beds_sus"] = resp["icu_beds_sus"].fillna(0)
resp["total_beds"] = resp["total_beds"].fillna(0)
resp["has_icu"] = (resp["icu_beds_sus"] > 0).astype(int)
resp["age_group"] = pd.cut(resp["age"], bins=[0, 1, 18, 40, 60, 75, 120],
                            labels=["<1", "1-17", "18-39", "40-59", "60-74", "75+"])

print(f"Total admissions: {len(resp):,}")
print(f"Hospitals: {resp['CNES'].nunique()}")
print(f"Hospital names loaded: {len(names_df)}")

Total admissions: 116,374
Hospitals: 562
Hospital names loaded: 486


## 1. Variação Bruta em Mortalidade Hospitalar

In [2]:
# Hospital-level stats
hosp = resp.groupby("CNES").agg(
    n=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
    pct_emergency=("is_emergency", "mean"),
    pct_male=("is_male", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    mean_cost=("VAL_TOT", "mean"),
    pct_icu_used=("icu_used", "mean"),
    icu_beds=("icu_beds_sus", "first"),
    total_beds=("total_beds", "first"),
    icu_level=("icu_capacity_level", "first"),
    nat_jur=("nat_jur_category", "first"),
    broad_type=("broad_type", "first"),
).reset_index()

hosp["nome"] = hosp["CNES"].apply(lambda c: hospital_name(c, names_df))
hosp["annual_volume"] = hosp["n"] / resp["year"].nunique()

# Volume tiers
hosp["volume_tier"] = pd.cut(hosp["annual_volume"],
    bins=[0, 10, 30, 100, 500, 10000],
    labels=["<10/yr", "10-30/yr", "30-100/yr", "100-500/yr", ">500/yr"])

# Restrict to hospitals with n >= 30 for meaningful analysis
hosp_sig = hosp[hosp["n"] >= 30].copy()

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# A: Histogram of hospital mortality rates
ax = axes[0]
ax.hist(hosp_sig["mortality"] * 100, bins=40, color=COLORS["primary"], alpha=0.7, edgecolor="white")
ax.axvline(resp["MORTE"].mean() * 100, color=COLORS["danger"], linestyle="--", linewidth=2,
           label=f"Média SP: {resp['MORTE'].mean()*100:.1f}%")
ax.set_xlabel("Mortalidade (%)")
ax.set_ylabel("N hospitais")
ax.set_title(f"A. Distribuição da Mortalidade (n≥30, {len(hosp_sig)} hosp.)", fontweight="bold")
ax.legend()

# B: Mortality by volume tier
ax = axes[1]
tier_stats = hosp_sig.groupby("volume_tier", observed=True).agg(
    median_mort=("mortality", "median"),
    q25=("mortality", lambda x: x.quantile(0.25)),
    q75=("mortality", lambda x: x.quantile(0.75)),
    n_hosp=("CNES", "count"),
).reset_index()
x_pos = range(len(tier_stats))
ax.bar(x_pos, tier_stats["median_mort"] * 100, color=COLORS["primary"], alpha=0.7)
ax.errorbar(x_pos, tier_stats["median_mort"] * 100,
            yerr=[(tier_stats["median_mort"] - tier_stats["q25"]) * 100,
                  (tier_stats["q75"] - tier_stats["median_mort"]) * 100],
            fmt="none", color="black", capsize=5)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{t}\n(n={n})" for t, n in zip(tier_stats["volume_tier"], tier_stats["n_hosp"])], fontsize=9)
ax.set_ylabel("Mortalidade mediana (%)")
ax.set_title("B. Volume Anual × Mortalidade", fontweight="bold")

# C: Scatter volume vs mortality
ax = axes[2]
ax.scatter(hosp_sig["annual_volume"], hosp_sig["mortality"] * 100,
           s=hosp_sig["mean_age"], alpha=0.4, c=COLORS["primary"], edgecolors="white", linewidth=0.3)
ax.set_xscale("log")
ax.set_xlabel("Volume anual (log)")
ax.set_ylabel("Mortalidade (%)")
ax.set_title("C. Volume × Mortalidade (tamanho = idade)", fontweight="bold")
ax.axhline(resp["MORTE"].mean() * 100, color=COLORS["danger"], linestyle="--", alpha=0.5)

r_vol, p_vol = stats.spearmanr(hosp_sig["annual_volume"], hosp_sig["mortality"])
ax.text(0.05, 0.95, f"Spearman ρ = {r_vol:.3f}\np = {p_vol:.2e}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.suptitle("Variação Hospitalar em Mortalidade J96", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "hospital_variation", prefix="06")

print(f"Hospitais com n≥30: {len(hosp_sig)}")
print(f"Mortalidade: mediana {hosp_sig['mortality'].median()*100:.1f}%, "
      f"IQR [{hosp_sig['mortality'].quantile(0.25)*100:.1f}% – {hosp_sig['mortality'].quantile(0.75)*100:.1f}%]")
print(f"Volume-outcome (Spearman): ρ = {r_vol:.3f}, p = {p_vol:.2e}")

  Saved: 06_hospital_variation.png
Hospitais com n≥30: 351
Mortalidade: mediana 37.8%, IQR [24.9% – 52.2%]
Volume-outcome (Spearman): ρ = -0.084, p = 1.17e-01


## 2. Ajuste de Risco: Mortalidade Esperada vs Observada

Para separar performance hospitalar de case-mix, calculamos a mortalidade **esperada** para cada hospital baseada no perfil de seus pacientes (idade, sexo, tipo de admissão). A razão observado/esperado (O/E) indica se o hospital mata mais ou menos do que seria esperado dado seus pacientes.

In [3]:
# Risk adjustment: compute expected mortality per patient based on age, sex, admission type
# Use state-wide rates as reference
strata_cols = ["age_group", "is_male", "is_emergency"]
strata_rates = resp.groupby(strata_cols, observed=True)["MORTE"].mean().reset_index()
strata_rates.columns = [*strata_cols, "expected_mortality"]

resp_adj = resp.merge(strata_rates, on=strata_cols, how="left")
resp_adj["expected_mortality"] = resp_adj["expected_mortality"].fillna(resp["MORTE"].mean())

# Hospital-level: sum expected deaths, compute O/E ratio
hosp_adj = resp_adj.groupby("CNES").agg(
    n=("MORTE", "count"),
    observed_deaths=("MORTE", "sum"),
    expected_deaths=("expected_mortality", "sum"),
    observed_mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
    pct_emergency=("is_emergency", "mean"),
).reset_index()

hosp_adj["expected_mortality_rate"] = hosp_adj["expected_deaths"] / hosp_adj["n"]
hosp_adj["oe_ratio"] = hosp_adj["observed_deaths"] / hosp_adj["expected_deaths"].clip(lower=0.5)
hosp_adj["excess_deaths"] = hosp_adj["observed_deaths"] - hosp_adj["expected_deaths"]

# Merge with hospital info
hosp_adj = hosp_adj.merge(hosp[["CNES", "nome", "icu_beds", "total_beds", "icu_level", "nat_jur",
                                 "broad_type", "annual_volume", "volume_tier"]], on="CNES", how="left")

hosp_adj_sig = hosp_adj[hosp_adj["n"] >= 50].copy()

# Wilson confidence interval for O/E
from scipy.stats import norm
z = 1.96
hosp_adj_sig["oe_se"] = np.sqrt(hosp_adj_sig["observed_deaths"]) / hosp_adj_sig["expected_deaths"].clip(lower=0.5)
hosp_adj_sig["oe_lower"] = hosp_adj_sig["oe_ratio"] - z * hosp_adj_sig["oe_se"]
hosp_adj_sig["oe_upper"] = hosp_adj_sig["oe_ratio"] + z * hosp_adj_sig["oe_se"]

# Classify: significantly above 1 = underperformer, below 1 = overperformer
hosp_adj_sig["performance"] = "Esperado"
hosp_adj_sig.loc[hosp_adj_sig["oe_lower"] > 1.0, "performance"] = "Acima do Esperado (pior)"
hosp_adj_sig.loc[hosp_adj_sig["oe_upper"] < 1.0, "performance"] = "Abaixo do Esperado (melhor)"

perf_counts = hosp_adj_sig["performance"].value_counts()
n_sig = len(hosp_adj_sig)

print(f"Hospitais com n≥50: {n_sig}")
print(f"\nClassificação de Performance (IC 95% do O/E):")
for perf, cnt in perf_counts.items():
    print(f"  {perf}: {cnt} ({cnt/n_sig*100:.0f}%)")

print(f"\nO/E médio: {hosp_adj_sig['oe_ratio'].mean():.3f}")
print(f"O/E mediano: {hosp_adj_sig['oe_ratio'].median():.3f}")

Hospitais com n≥50: 306

Classificação de Performance (IC 95% do O/E):
  Esperado: 140 (46%)
  Abaixo do Esperado (melhor): 91 (30%)
  Acima do Esperado (pior): 75 (25%)

O/E médio: 1.016
O/E mediano: 0.989


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# A: O/E distribution
ax = axes[0]
colors_perf = {"Abaixo do Esperado (melhor)": COLORS["success"],
               "Esperado": COLORS["muted"],
               "Acima do Esperado (pior)": COLORS["danger"]}
for perf, color in colors_perf.items():
    subset = hosp_adj_sig[hosp_adj_sig["performance"] == perf]
    ax.hist(subset["oe_ratio"], bins=20, color=color, alpha=0.6, label=f"{perf} ({len(subset)})")
ax.axvline(1.0, color="black", linestyle="--", linewidth=1.5, label="Esperado = 1.0")
ax.set_xlabel("Razão O/E")
ax.set_ylabel("N hospitais")
ax.set_title("A. Distribuição da Razão O/E", fontweight="bold")
ax.legend(fontsize=8)

# B: Funnel plot — O/E vs volume
ax = axes[1]
for perf, color in colors_perf.items():
    subset = hosp_adj_sig[hosp_adj_sig["performance"] == perf]
    ax.scatter(subset["n"], subset["oe_ratio"],
              s=40, alpha=0.5, c=color, edgecolors="white", linewidth=0.3, label=f"{perf}")

# Funnel limits
n_range = np.arange(50, hosp_adj_sig["n"].max() + 1)
p_mean = resp["MORTE"].mean()
se_funnel = 1 / np.sqrt(n_range * p_mean)
ax.plot(n_range, 1 + z * se_funnel, "--", color="gray", alpha=0.5, linewidth=0.8)
ax.plot(n_range, 1 - z * se_funnel, "--", color="gray", alpha=0.5, linewidth=0.8)
ax.axhline(1.0, color="black", linestyle="-", linewidth=0.5)
ax.set_xscale("log")
ax.set_xlabel("Volume total (log)")
ax.set_ylabel("Razão O/E")
ax.set_title("B. Funnel Plot: O/E vs Volume", fontweight="bold")
ax.set_ylim(0, 3.5)

# Label extreme outliers
extremes = hosp_adj_sig[(hosp_adj_sig["oe_ratio"] > 2.0) & (hosp_adj_sig["n"] > 100)]
for _, r in extremes.iterrows():
    ax.annotate(r["nome"][:20], (r["n"], r["oe_ratio"]),
                fontsize=6, textcoords="offset points", xytext=(5, 3))

# C: Observed vs Expected scatter
ax = axes[2]
ax.scatter(hosp_adj_sig["expected_mortality_rate"] * 100,
           hosp_adj_sig["observed_mortality"] * 100,
           s=hosp_adj_sig["n"] / 5, alpha=0.4, c=COLORS["primary"], edgecolors="white", linewidth=0.3)
lims = [0, 100]
ax.plot(lims, lims, "--", color=COLORS["danger"], alpha=0.5, label="Linha identidade")
ax.set_xlabel("Mortalidade Esperada (%)")
ax.set_ylabel("Mortalidade Observada (%)")
ax.set_title("C. Observado × Esperado", fontweight="bold")
ax.legend(fontsize=8)
ax.set_xlim(0, 85)
ax.set_ylim(0, 100)

fig.suptitle("Ajuste de Risco: Mortalidade Observada vs Esperada por Hospital",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "risk_adjustment", prefix="06")

  Saved: 06_risk_adjustment.png


## 3. Top 20 Melhores e Piores Hospitais (Ajustados)

In [5]:
# Best performers (lowest O/E, n >= 100)
best = hosp_adj_sig[hosp_adj_sig["n"] >= 100].nsmallest(20, "oe_ratio")
worst = hosp_adj_sig[hosp_adj_sig["n"] >= 100].nlargest(20, "oe_ratio")

fig, axes = plt.subplots(1, 2, figsize=(22, 10))

# Left: Best
ax = axes[0]
y_pos = range(len(best))
ax.barh(y_pos, best["oe_ratio"], color=COLORS["success"], alpha=0.8)
ax.errorbar(best["oe_ratio"], y_pos,
            xerr=z * best["oe_se"], fmt="none", color="black", capsize=3)
ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{n[:30]} (n={v})" for n, v in zip(best["nome"], best["n"])], fontsize=8)
ax.set_xlabel("Razão O/E")
ax.set_title("Top 20 Melhores (menor O/E, n≥100)", fontweight="bold")
ax.invert_yaxis()

# Right: Worst
ax = axes[1]
y_pos = range(len(worst))
ax.barh(y_pos, worst["oe_ratio"], color=COLORS["danger"], alpha=0.8)
ax.errorbar(worst["oe_ratio"], y_pos,
            xerr=z * worst["oe_se"], fmt="none", color="black", capsize=3)
ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{n[:30]} (n={v})" for n, v in zip(worst["nome"], worst["n"])], fontsize=8)
ax.set_xlabel("Razão O/E")
ax.set_title("Top 20 Piores (maior O/E, n≥100)", fontweight="bold")
ax.invert_yaxis()

fig.suptitle("Ranking Hospitalar Ajustado por Risco (O/E)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "hospital_ranking", prefix="06")

print("=" * 100)
print("TOP 20 MELHORES HOSPITAIS (menor O/E, n≥100)")
print("=" * 100)
print(f"{'#':>3} {'Hospital':<40} {'N':>6} {'Obs%':>6} {'Esp%':>6} {'O/E':>6} {'Exc.':>6} {'UTI':>5} {'Nat.':>8}")
for rank, (_, r) in enumerate(best.iterrows(), 1):
    print(f"{rank:>3}. {r['nome'][:38]:<38} {r['n']:>5} {r['observed_mortality']*100:>5.1f} "
          f"{r['expected_mortality_rate']*100:>5.1f} {r['oe_ratio']:>5.2f} {r['excess_deaths']:>+5.0f} "
          f"{r['icu_beds']:>4.0f} {str(r['nat_jur'])[:7]:>8}")

print(f"\n{'=' * 100}")
print("TOP 20 PIORES HOSPITAIS (maior O/E, n≥100)")
print("=" * 100)
print(f"{'#':>3} {'Hospital':<40} {'N':>6} {'Obs%':>6} {'Esp%':>6} {'O/E':>6} {'Exc.':>6} {'UTI':>5} {'Nat.':>8}")
for rank, (_, r) in enumerate(worst.iterrows(), 1):
    print(f"{rank:>3}. {r['nome'][:38]:<38} {r['n']:>5} {r['observed_mortality']*100:>5.1f} "
          f"{r['expected_mortality_rate']*100:>5.1f} {r['oe_ratio']:>5.2f} {r['excess_deaths']:>+5.0f} "
          f"{r['icu_beds']:>4.0f} {str(r['nat_jur'])[:7]:>8}")

  Saved: 06_hospital_ranking.png
TOP 20 MELHORES HOSPITAIS (menor O/E, n≥100)
  # Hospital                                      N   Obs%   Esp%    O/E   Exc.   UTI     Nat.
  1. 2082179                                  173   0.0   3.7  0.00    -6    0  assoc_p
  2. 0102040                                  362   0.3  39.5  0.01  -142    0  unknown
  3. 0109746                                  932   0.6  41.9  0.02  -385    0  unknown
  4. 0134163                                  333   3.9  49.9  0.08  -153    0  unknown
  5. HOSPITAL DE BASE DE SAO JOSE DO RIO PR   577   1.9  24.3  0.08  -129    5  fundaca
  6. 2790564                                  203   1.0   5.4  0.18    -9    0   public
  7. HOSPITAL DR RENATO SILVA DE SOCORRO      337  12.5  35.7  0.35   -78    0  assoc_p
  8. CASA DE CARIDADE SAO VICENTE DE PAULO    310  12.9  37.0  0.35   -75    0  assoc_p
  9. FUNDACAO PIO XII BARRETOS                170  14.7  41.0  0.36   -45   17  fundaca
 10. HOSPITAL MUNICIPAL DE PAULINIA

## 4. Relação Volume-Desfecho

In [6]:
# Volume-outcome: does treating more J96 patients improve risk-adjusted outcomes?
vol_oe = hosp_adj_sig.copy()
r_vol_oe, p_vol_oe = stats.spearmanr(vol_oe["annual_volume"], vol_oe["oe_ratio"])

# Compare by volume tier
vol_tier_oe = vol_oe.groupby("volume_tier", observed=True).agg(
    median_oe=("oe_ratio", "median"),
    mean_oe=("oe_ratio", "mean"),
    n_hosp=("CNES", "count"),
    total_patients=("n", "sum"),
    total_excess=("excess_deaths", "sum"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.scatter(vol_oe["annual_volume"], vol_oe["oe_ratio"],
           s=vol_oe["n"] / 10, alpha=0.4, c=COLORS["primary"], edgecolors="white", linewidth=0.3)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("Volume anual (log)")
ax.set_ylabel("Razão O/E")
ax.set_title(f"A. Volume × O/E (ρ = {r_vol_oe:.3f}, p = {p_vol_oe:.2e})", fontweight="bold")
ax.set_ylim(0, 3.5)

ax = axes[1]
x_pos = range(len(vol_tier_oe))
ax.bar(x_pos, vol_tier_oe["median_oe"], color=COLORS["primary"], alpha=0.7)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{t}\n({n} hosp.)" for t, n in zip(vol_tier_oe["volume_tier"], vol_tier_oe["n_hosp"])], fontsize=9)
ax.set_ylabel("Mediana O/E")
ax.set_title("B. O/E Mediano por Faixa de Volume", fontweight="bold")
for i, (_, r) in enumerate(vol_tier_oe.iterrows()):
    ax.text(i, r["median_oe"] + 0.02, f"{r['median_oe']:.2f}", ha="center", fontsize=10, fontweight="bold")

fig.suptitle("Relação Volume-Desfecho (ajustado por risco)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "volume_outcome", prefix="06")

print(f"Volume-Outcome (Spearman): ρ = {r_vol_oe:.3f}, p = {p_vol_oe:.2e}")
print(f"\nO/E mediano por faixa de volume:")
print(vol_tier_oe[["volume_tier", "n_hosp", "median_oe", "total_excess"]].to_string(index=False))

  Saved: 06_volume_outcome.png
Volume-Outcome (Spearman): ρ = 0.031, p = 5.89e-01

O/E mediano por faixa de volume:
volume_tier  n_hosp  median_oe  total_excess
     <10/yr      71   0.992116   -155.802189
   10-30/yr     119   0.995897    525.760881
  30-100/yr      98   0.995748   1096.615056
 100-500/yr      18   0.789566  -1097.380262


## 5. Características Hospitalares que Predizem Performance

In [7]:
# O/E by hospital characteristics
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# A: By ICU capacity level
ax = axes[0, 0]
icu_perf = hosp_adj_sig.groupby("icu_level", observed=True).agg(
    median_oe=("oe_ratio", "median"),
    n_hosp=("CNES", "count"),
    total_excess=("excess_deaths", "sum"),
).reindex(["no_icu", "small_icu", "medium_icu", "large_icu"])
icu_perf = icu_perf.dropna()
icu_labels = {"no_icu": "Sem UTI", "small_icu": "Peq. (1-10)",
              "medium_icu": "Méd. (11-30)", "large_icu": "Grande (>30)"}
x_pos = range(len(icu_perf))
ax.bar(x_pos, icu_perf["median_oe"], color=[COLORS["danger"], COLORS["warning"], COLORS["primary"], COLORS["success"]], alpha=0.7)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{icu_labels.get(l, l)}\n({n})" for l, n in zip(icu_perf.index, icu_perf["n_hosp"])], fontsize=9)
ax.set_ylabel("Mediana O/E")
ax.set_title("A. O/E por Capacidade de UTI", fontweight="bold")

# B: By legal nature
ax = axes[0, 1]
nat_perf = hosp_adj_sig.groupby("nat_jur", observed=True).agg(
    median_oe=("oe_ratio", "median"),
    n_hosp=("CNES", "count"),
).sort_values("median_oe")
nat_perf = nat_perf[nat_perf["n_hosp"] >= 3]
x_pos = range(len(nat_perf))
ax.barh(x_pos, nat_perf["median_oe"], color=COLORS["primary"], alpha=0.7)
ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_yticks(x_pos)
ax.set_yticklabels([f"{n} ({c})" for n, c in zip(nat_perf.index, nat_perf["n_hosp"])], fontsize=9)
ax.set_xlabel("Mediana O/E")
ax.set_title("B. O/E por Natureza Jurídica", fontweight="bold")
ax.invert_yaxis()

# C: By broad type
ax = axes[1, 0]
type_perf = hosp_adj_sig.groupby("broad_type", observed=True).agg(
    median_oe=("oe_ratio", "median"),
    n_hosp=("CNES", "count"),
).sort_values("median_oe")
type_perf = type_perf[type_perf["n_hosp"] >= 3]
x_pos = range(len(type_perf))
ax.barh(x_pos, type_perf["median_oe"], color=COLORS["secondary"], alpha=0.7)
ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_yticks(x_pos)
ax.set_yticklabels([f"{t} ({c})" for t, c in zip(type_perf.index, type_perf["n_hosp"])], fontsize=9)
ax.set_xlabel("Mediana O/E")
ax.set_title("C. O/E por Tipo de Estabelecimento", fontweight="bold")
ax.invert_yaxis()

# D: ICU beds vs O/E scatter
ax = axes[1, 1]
has_icu_data = hosp_adj_sig[hosp_adj_sig["icu_beds"] > 0]
ax.scatter(has_icu_data["icu_beds"], has_icu_data["oe_ratio"],
           s=has_icu_data["n"] / 10, alpha=0.5, c=COLORS["primary"], edgecolors="white", linewidth=0.3)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Leitos UTI SUS")
ax.set_ylabel("Razão O/E")
if len(has_icu_data) > 5:
    r_icu_oe, p_icu_oe = stats.spearmanr(has_icu_data["icu_beds"], has_icu_data["oe_ratio"])
    ax.set_title(f"D. Leitos UTI × O/E (ρ = {r_icu_oe:.3f}, p = {p_icu_oe:.2e})", fontweight="bold")
else:
    ax.set_title("D. Leitos UTI × O/E", fontweight="bold")

fig.suptitle("Determinantes de Performance Hospitalar", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "hospital_determinants", prefix="06")

print("O/E mediano por capacidade de UTI:")
print(icu_perf[["median_oe", "n_hosp", "total_excess"]].to_string())
print(f"\nO/E mediano por natureza jurídica:")
print(nat_perf[["median_oe", "n_hosp"]].to_string())

  Saved: 06_hospital_determinants.png
O/E mediano por capacidade de UTI:
            median_oe  n_hosp  total_excess
icu_level                                  
no_icu       0.991318     220    608.982354
small_icu    0.991553      59    213.584733
medium_icu   1.032786      24   -334.103438
large_icu    0.874554       3   -119.270162

O/E mediano por natureza jurídica:
                  median_oe  n_hosp
nat_jur                            
unknown            0.175584       8
fundacao_privada   0.774768      18
assoc_privada      0.954409     140
public             1.097400     139


## 6. Consistência Temporal: Quem Sempre Performa Bem/Mal?

In [8]:
# Consistency: compute O/E per year per hospital
yearly_strata = resp.groupby([*strata_cols, resp["year"].astype(int)], observed=True)["MORTE"].mean().reset_index()
yearly_strata.columns = [*strata_cols, "year", "expected_mortality_yr"]

resp_yr = resp.copy()
resp_yr["year_int"] = resp_yr["year"].astype(int)
resp_yr = resp_yr.merge(yearly_strata, left_on=[*strata_cols, "year_int"],
                        right_on=[*strata_cols, "year"], how="left", suffixes=("", "_yr"))
resp_yr["expected_mortality_yr"] = resp_yr["expected_mortality_yr"].fillna(resp["MORTE"].mean())

hosp_yr = resp_yr.groupby(["CNES", "year_int"]).agg(
    n=("MORTE", "count"),
    obs_deaths=("MORTE", "sum"),
    exp_deaths=("expected_mortality_yr", "sum"),
).reset_index()
hosp_yr["oe_yr"] = hosp_yr["obs_deaths"] / hosp_yr["exp_deaths"].clip(lower=0.5)

# Hospitals with data in ≥8 years and n≥20/year
eligible = hosp_yr[hosp_yr["n"] >= 20].groupby("CNES").size()
consistent_cnes = eligible[eligible >= 8].index

hosp_consistent = hosp_yr[hosp_yr["CNES"].isin(consistent_cnes)].copy()

# Compute coefficient of variation of O/E and mean O/E
consistency = hosp_consistent.groupby("CNES").agg(
    mean_oe=("oe_yr", "mean"),
    std_oe=("oe_yr", "std"),
    n_years=("year_int", "count"),
    total_n=("n", "sum"),
).reset_index()
consistency["cv_oe"] = consistency["std_oe"] / consistency["mean_oe"]
consistency["nome"] = consistency["CNES"].apply(lambda c: hospital_name(c, names_df))

# Consistently good: mean O/E < 0.8 and CV < 0.3
consistently_good = consistency[(consistency["mean_oe"] < 0.8) & (consistency["cv_oe"] < 0.3) & (consistency["total_n"] >= 200)]
consistently_bad = consistency[(consistency["mean_oe"] > 1.3) & (consistency["cv_oe"] < 0.4) & (consistency["total_n"] >= 200)]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# A: Mean O/E vs CV scatter
ax = axes[0]
ax.scatter(consistency["mean_oe"], consistency["cv_oe"],
           s=consistency["total_n"] / 20, alpha=0.5, c=COLORS["primary"], edgecolors="white", linewidth=0.3)
ax.axvline(1.0, color="black", linestyle="--", linewidth=1)
ax.axhline(0.3, color="gray", linestyle=":", linewidth=0.5)
# Highlight consistently good/bad
if len(consistently_good) > 0:
    ax.scatter(consistently_good["mean_oe"], consistently_good["cv_oe"],
               s=consistently_good["total_n"] / 20, alpha=0.8, c=COLORS["success"],
               edgecolors="black", linewidth=1, label=f"Consist. bons ({len(consistently_good)})")
if len(consistently_bad) > 0:
    ax.scatter(consistently_bad["mean_oe"], consistently_bad["cv_oe"],
               s=consistently_bad["total_n"] / 20, alpha=0.8, c=COLORS["danger"],
               edgecolors="black", linewidth=1, label=f"Consist. ruins ({len(consistently_bad)})")
ax.set_xlabel("O/E Médio (8+ anos)")
ax.set_ylabel("Coeficiente de Variação")
ax.set_title("A. Consistência da Performance (n≥20/ano, ≥8 anos)", fontweight="bold")
ax.legend(fontsize=8)

# B: Time series for top 5 consistently good and bad
ax = axes[1]
top_good = consistently_good.nlargest(3, "total_n") if len(consistently_good) >= 3 else consistently_good
top_bad = consistently_bad.nlargest(3, "total_n") if len(consistently_bad) >= 3 else consistently_bad

for _, r in top_good.iterrows():
    yr_data = hosp_yr[(hosp_yr["CNES"] == r["CNES"]) & (hosp_yr["n"] >= 20)].sort_values("year_int")
    ax.plot(yr_data["year_int"], yr_data["oe_yr"], "o-", linewidth=2, markersize=5,
            color=COLORS["success"], label=f"✓ {r['nome'][:20]}")
for _, r in top_bad.iterrows():
    yr_data = hosp_yr[(hosp_yr["CNES"] == r["CNES"]) & (hosp_yr["n"] >= 20)].sort_values("year_int")
    ax.plot(yr_data["year_int"], yr_data["oe_yr"], "s--", linewidth=2, markersize=5,
            color=COLORS["danger"], label=f"✗ {r['nome'][:20]}")

ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax.set_xlabel("Ano")
ax.set_ylabel("Razão O/E")
ax.set_title("B. Trajetória dos Hospitais Mais Consistentes", fontweight="bold")
ax.legend(fontsize=7, loc="upper left")

fig.suptitle("Consistência Temporal da Performance Hospitalar",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "consistency", prefix="06")

print(f"Hospitais com ≥8 anos e n≥20/ano: {len(consistency)}")
print(f"Consistentemente bons (O/E<0.8, CV<0.3, N≥200): {len(consistently_good)}")
print(f"Consistentemente ruins (O/E>1.3, CV<0.4, N≥200): {len(consistently_bad)}")

if len(consistently_good) > 0:
    print(f"\nConsistentemente BONS:")
    for _, r in consistently_good.sort_values("mean_oe").iterrows():
        print(f"  {r['nome'][:35]:<35} O/E={r['mean_oe']:.2f} CV={r['cv_oe']:.2f} N={r['total_n']}")

if len(consistently_bad) > 0:
    print(f"\nConsistentemente RUINS:")
    for _, r in consistently_bad.sort_values("mean_oe", ascending=False).iterrows():
        print(f"  {r['nome'][:35]:<35} O/E={r['mean_oe']:.2f} CV={r['cv_oe']:.2f} N={r['total_n']}")

  Saved: 06_consistency.png
Hospitais com ≥8 anos e n≥20/ano: 97
Consistentemente bons (O/E<0.8, CV<0.3, N≥200): 4
Consistentemente ruins (O/E>1.3, CV<0.4, N≥200): 22

Consistentemente BONS:
  HOSPITAL DAS CLINICAS DA FACULDADE  O/E=0.54 CV=0.29 N=4214
  HC DA FMUSP HOSPITAL DAS CLINICAS S O/E=0.70 CV=0.18 N=1210
  HOSPITAL AUGUSTO DE OLIVEIRA CAMARG O/E=0.73 CV=0.30 N=1064
  COMPLEXO HOSPITALAR PREFEITO EDIVAL O/E=0.75 CV=0.23 N=2665

Consistentemente RUINS:
  HOSPITAL HELIOPOLIS UNIDADE DE GEST O/E=1.72 CV=0.15 N=441
  HOSPITAL GERAL DE VILA NOVA CACHOEI O/E=1.68 CV=0.14 N=364
  HOSP MUN JABAQUARA ARTUR RIBEIRO DE O/E=1.66 CV=0.11 N=1011
  HOSP MUN IGNACIO PROENCA DE GOUVEA  O/E=1.63 CV=0.09 N=602
  HOSPITAL DE CLINICAS DR RADAMES NAR O/E=1.59 CV=0.14 N=387
  HOSP MUN VER JOSE STOROPOLLI        O/E=1.59 CV=0.24 N=796
  HOSPITAL GERAL DE GUARULHOS PROF DR O/E=1.59 CV=0.12 N=789
  HOSPITAL GERAL DE ITAPEVI           O/E=1.59 CV=0.23 N=798
  SANTA CASA DE LIMEIRA               O/E=1.59 

/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_42654/1716012656.py:83: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_42654/1716012656.py:83: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) Arial.
  plt.tight_layout()
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/experiments/respiratory_failure/notebooks/shared.py:178: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  fig.savefig(plot_dir / fname, bbox_inches="tight", facecolor="white")
/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/experiments/respiratory_failure/notebooks/shared.py:178: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) Arial.
  fig.savefig(plot_dir / fname, bbox_inches="tight", facecolor="white")


## 7. Custo vs Performance: Quem Gasta Mais Salva Mais?

In [9]:
# Cost-performance analysis
cost_perf = hosp_adj_sig.merge(
    hosp[["CNES", "mean_cost", "mean_los", "pct_icu_used"]], on="CNES", how="left"
)

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# A: Cost vs O/E
ax = axes[0]
ax.scatter(cost_perf["mean_cost"], cost_perf["oe_ratio"],
           s=cost_perf["n"] / 10, alpha=0.4, c=COLORS["primary"], edgecolors="white", linewidth=0.3)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
r_cost_oe, p_cost_oe = stats.spearmanr(cost_perf["mean_cost"].dropna(), cost_perf.loc[cost_perf["mean_cost"].notna(), "oe_ratio"])
ax.set_xlabel("Custo Médio por Internação (R$)")
ax.set_ylabel("Razão O/E")
ax.set_title(f"A. Custo × Performance (ρ = {r_cost_oe:.3f})", fontweight="bold")

# B: LOS vs O/E
ax = axes[1]
ax.scatter(cost_perf["mean_los"], cost_perf["oe_ratio"],
           s=cost_perf["n"] / 10, alpha=0.4, c=COLORS["secondary"], edgecolors="white", linewidth=0.3)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
r_los_oe, p_los_oe = stats.spearmanr(cost_perf["mean_los"].dropna(), cost_perf.loc[cost_perf["mean_los"].notna(), "oe_ratio"])
ax.set_xlabel("Permanência Média (dias)")
ax.set_ylabel("Razão O/E")
ax.set_title(f"B. Permanência × Performance (ρ = {r_los_oe:.3f})", fontweight="bold")

# C: ICU usage vs O/E
ax = axes[2]
ax.scatter(cost_perf["pct_icu_used"] * 100, cost_perf["oe_ratio"],
           s=cost_perf["n"] / 10, alpha=0.4, c=COLORS["success"], edgecolors="white", linewidth=0.3)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
r_icu_perf, p_icu_perf = stats.spearmanr(cost_perf["pct_icu_used"].dropna(), cost_perf.loc[cost_perf["pct_icu_used"].notna(), "oe_ratio"])
ax.set_xlabel("% Pacientes com UTI")
ax.set_ylabel("Razão O/E")
ax.set_title(f"C. Uso de UTI × Performance (ρ = {r_icu_perf:.3f})", fontweight="bold")

fig.suptitle("Custo e Processos de Cuidado vs Performance Ajustada",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "cost_performance", prefix="06")

print(f"Correlações (Spearman) com O/E:")
print(f"  Custo médio: ρ = {r_cost_oe:.3f}, p = {p_cost_oe:.2e}")
print(f"  Permanência: ρ = {r_los_oe:.3f}, p = {p_los_oe:.2e}")
print(f"  % UTI usada: ρ = {r_icu_perf:.3f}, p = {p_icu_perf:.2e}")

/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_42654/1816897682.py:33: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r_icu_perf, p_icu_perf = stats.spearmanr(cost_perf["pct_icu_used"].dropna(), cost_perf.loc[cost_perf["pct_icu_used"].notna(), "oe_ratio"])


  Saved: 06_cost_performance.png
Correlações (Spearman) com O/E:
  Custo médio: ρ = 0.130, p = 2.27e-02
  Permanência: ρ = 0.155, p = 6.58e-03
  % UTI usada: ρ = nan, p = nan


## 8. Excesso de Mortes: Impacto Absoluto dos Hospitais Sub-Performantes

In [10]:
# Absolute impact: which hospitals contribute most excess deaths?
excess = hosp_adj_sig[hosp_adj_sig["excess_deaths"] > 10].sort_values("excess_deaths", ascending=False)

fig, ax = plt.subplots(figsize=(14, 10))
top_excess = excess.head(25)
y_pos = range(len(top_excess))
colors = [COLORS["danger"] if r["oe_lower"] > 1.0 else COLORS["warning"] for _, r in top_excess.iterrows()]
ax.barh(y_pos, top_excess["excess_deaths"], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{n[:30]} (O/E={oe:.2f})" for n, oe in zip(top_excess["nome"], top_excess["oe_ratio"])], fontsize=8)
ax.set_xlabel("Excesso de Óbitos (Observado − Esperado)")
ax.set_title("Top 25 Hospitais com Maior Excesso de Mortes", fontweight="bold", fontsize=13)
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=COLORS["danger"], label="Significativamente acima (IC 95%)"),
                   Patch(facecolor=COLORS["warning"], label="Acima, mas não significativo")],
          loc="lower right", fontsize=9)

plt.tight_layout()
save_plot(fig, "excess_deaths", prefix="06")

total_excess = hosp_adj_sig[hosp_adj_sig["excess_deaths"] > 0]["excess_deaths"].sum()
total_saved = abs(hosp_adj_sig[hosp_adj_sig["excess_deaths"] < 0]["excess_deaths"].sum())

print(f"Total excesso de óbitos (hospitais acima do esperado): {total_excess:,.0f}")
print(f"Total vidas 'salvas' (hospitais abaixo do esperado): {total_saved:,.0f}")
print(f"Balanço líquido: {total_excess - total_saved:+,.0f}")
print(f"\nTop 10 hospitais com maior excesso:")
for _, r in excess.head(10).iterrows():
    print(f"  {r['nome'][:35]:<35} Exc: {r['excess_deaths']:>+6.0f}  O/E: {r['oe_ratio']:.2f}  N: {r['n']}")

  Saved: 06_excess_deaths.png
Total excesso de óbitos (hospitais acima do esperado): 5,741
Total vidas 'salvas' (hospitais abaixo do esperado): 5,372
Balanço líquido: +369

Top 10 hospitais com maior excesso:
  HOSP MUN JABAQUARA ARTUR RIBEIRO DE Exc:   +239  O/E: 1.62  N: 1011
  HOSP MUN VER JOSE STOROPOLLI        Exc:   +227  O/E: 1.62  N: 796
  HOSP ESCOLA EMILIO CARLOS CATANDUVA Exc:   +211  O/E: 1.68  N: 589
  HOSPITAL GERAL DE GUARULHOS PROF DR Exc:   +210  O/E: 1.67  N: 789
  CONJUNTO HOSPITALAR DO MANDAQUI SAO Exc:   +205  O/E: 1.49  N: 942
  INSTITUTO DO CANCER DO ESTADO DE SA Exc:   +202  O/E: 1.36  N: 1240
  HOSP MUN TIDE SETUBAL               Exc:   +202  O/E: 1.42  N: 970
  HOSP MUN IGNACIO PROENCA DE GOUVEA  Exc:   +199  O/E: 1.65  N: 602
  HOSPITAL HELIOPOLIS UNIDADE DE GEST Exc:   +149  O/E: 1.74  N: 441
  HOSPITAL DAS CLINICAS HCFAMEMA      Exc:   +139  O/E: 1.45  N: 968


## 9. Métricas Consolidadas

In [11]:
metrics = {
    "n_hospitals_analyzed": len(hosp_sig),
    "n_hospitals_risk_adjusted": len(hosp_adj_sig),
    "mortality_median_pct": round(hosp_sig["mortality"].median() * 100, 1),
    "mortality_iqr_low": round(hosp_sig["mortality"].quantile(0.25) * 100, 1),
    "mortality_iqr_high": round(hosp_sig["mortality"].quantile(0.75) * 100, 1),
    "volume_outcome_rho": round(r_vol, 3),
    "volume_outcome_oe_rho": round(r_vol_oe, 3),
    "oe_median": round(hosp_adj_sig["oe_ratio"].median(), 3),
    "n_above_expected": int(perf_counts.get("Acima do Esperado (pior)", 0)),
    "n_below_expected": int(perf_counts.get("Abaixo do Esperado (melhor)", 0)),
    "n_as_expected": int(perf_counts.get("Esperado", 0)),
    "pct_above": round(perf_counts.get("Acima do Esperado (pior)", 0) / n_sig * 100, 1),
    "total_excess_deaths": round(total_excess),
    "total_saved_lives": round(total_saved),
    "n_consistently_good": len(consistently_good),
    "n_consistently_bad": len(consistently_bad),
    "cost_oe_rho": round(r_cost_oe, 3),
    "los_oe_rho": round(r_los_oe, 3),
    "icu_usage_oe_rho": round(r_icu_perf, 3),
    "top_5_best": [
        {"nome": r["nome"], "oe_ratio": round(r["oe_ratio"], 2), "n": int(r["n"]),
         "excess_deaths": round(r["excess_deaths"])}
        for _, r in best.head(5).iterrows()
    ],
    "top_5_worst": [
        {"nome": r["nome"], "oe_ratio": round(r["oe_ratio"], 2), "n": int(r["n"]),
         "excess_deaths": round(r["excess_deaths"])}
        for _, r in worst.head(5).iterrows()
    ],
}
save_metrics(metrics, "06_hospital_performance")
print("Metrics saved.")

  Saved metrics: 06_hospital_performance.json
Metrics saved.
